# Kiểm tra và benchmark truy hồi RAG — Huế

Notebook này làm hai việc:

1. Thử tay luồng truy hồi: embed chunk, tìm theo câu hỏi, xem ngữ cảnh đưa vào LLM.
2. Chạy benchmark truy hồi trên bộ câu hỏi gán nhãn `eval/rag_queries.yml`, so dense (bge-m3) với BM25 và random.

Chuẩn bị (chạy trong terminal, ở thư mục gốc repo):

```
docker start <container postgres>        # DB ở localhost:5433
pip install -e .[embed]                  # sentence-transformers + torch
python -m pipeline migrate               # có migration 007 (cột embedding_model)
python -m pipeline download-model        # tải bge-m3 một lần (~4.5 GB tính cả bản safetensors)
python -m pipeline embed
```

In [1]:
from IPython.display import Markdown, display

from pipeline import db
from pipeline.embed import get_embedder, run_embed_chunks
from rag import HybridRetriever, RAGRetriever
from rag import benchmark

conn = db.connect()
print("Database:", conn.info.dbname)

Database: travel


## 1. Trạng thái index

Mỗi chunk ghi lại model đã sinh embedding. Retriever chỉ tìm trên chunk cùng model với query.

In [2]:
with conn.cursor() as cur:
    cur.execute(
        "SELECT COALESCE(embedding_model, '(chưa embed)'), COUNT(*)"
        " FROM place_chunks GROUP BY 1 ORDER BY 2 DESC"
    )
    for model, n in cur.fetchall():
        print(f"{model:30} {n:6d} chunk")

BAAI/bge-m3                      1574 chunk


## 2. Sinh embedding

`get_embedder()` đọc `EMBEDDING_MODEL`, mặc định `BAAI/bge-m3`. Không dùng `DeterministicEmbedder` ở đây: vector của nó là giá trị băm, không mang nghĩa, nên mọi kết quả tìm kiếm sẽ là ngẫu nhiên.

Chỉ chunk chưa có embedding của model hiện tại mới được sinh. Thêm `force=True` để sinh lại tất cả.

In [3]:
embedder = get_embedder()
count = run_embed_chunks(conn, embedder=embedder)
print(f"Đã sinh embedding cho {count} chunk bằng {embedder.model_name}")

c:\Users\baoba\OneDrive\Máy tính\Travel\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 14435.75it/s]


Đã sinh embedding cho 0 chunk bằng BAAI/bge-m3


In [4]:
retriever = RAGRetriever(conn, embedder=embedder)
total, embedded = retriever.index_status()
print(f"{embedded}/{total} chunk có embedding của {embedder.model_name}")

# Retriever agent dùng: dense + BM25 (giữ dấu, bỏ dấu, bigram bỏ dấu), ghép bằng RRF.
# Mặc định loại chunk của category không phải điểm đến (boi_canh, ha_tang). Dùng lại model đã tải.
hybrid = HybridRetriever.from_connection(conn, embedder=embedder)
print(hybrid.name)

1574/1574 chunk có embedding của BAAI/bge-m3
hybrid:BAAI/bge-m3+bm25_multi


## 3. Thử truy vấn

`hybrid` là retriever agent dùng. Với hybrid, `similarity` là điểm RRF (tổng 1/(60 + hạng) qua hai nhánh), không phải cosine.

In [5]:
def show(results, width=200):
    for i, c in enumerate(results, 1):
        print(f"[{i}] {c.place_name} ({c.category}) — similarity {c.similarity:.4f}")
        print(f"    {c.content[:width]}...")
        print(f"    Nguồn: {c.source_url}\n")


show(hybrid.search("Lịch sử và kiến trúc chùa Thiên Mụ", top_k=3))

[1] Chùa Thiên Mụ (chua) — similarity 0.0328
    == Lịch sử ==...
    Nguồn: https://vi.wikipedia.org/wiki/Chùa_Thiên_Mụ

[2] Chùa Thiên Mụ (chua) — similarity 0.0320
    == Kiến trúc ==...
    Nguồn: https://vi.wikipedia.org/wiki/Chùa_Thiên_Mụ

[3] Chùa Thiên Mụ (chua) — similarity 0.0313
    Chùa Thiên Mụ chính thức khởi lập năm Tân Sửu (1601), đời chúa Tiên - Nguyễn Hoàng. Dưới thời chúa Quốc - Nguyễn Phúc Chu (1691-1725) theo đà phát triển và hưng thịnh của Phật giáo xứ Đàng Trong, chùa...
    Nguồn: https://vi.wikipedia.org/wiki/Chùa_Thiên_Mụ



In [6]:
# Câu mô tả không nêu tên địa danh: dense phải hiểu nghĩa, BM25 thường trượt.
show(hybrid.search("Lăng của vị vua làm nhiều thơ, có hồ Lưu Khiêm", top_k=3))

[1] Lăng Tự Đức (lang_tam) — similarity 0.0328
    Bố cục khu lăng gồm 2 phần chính, bố trí trên 2 trục lăng mộ và tẩm điện dọc song song với nhau cùng lấy núi Giáng Khiêm ở phía trước làm tiền án, núi Dương Xuân làm hậu chẩm, hồ Lưu Khiêm làm yếu tố ...
    Nguồn: https://vi.wikipedia.org/wiki/Lăng_Tự_Đức

[2] Lăng Kiến Phúc (khac) — similarity 0.0315
    Lăng Kiến Phúc, hay Bồi Lăng (陪陵), là lăng tẩm của hoàng đế Kiến Phúc, vị hoàng đế thứ 7 của triều đại nhà Nguyễn trong lịch sử Việt Nam. Lăng tọa lạc tại phường Thủy Xuân, quận Thuận Hóa, thành phố H...
    Nguồn: https://vi.wikipedia.org/wiki/Lăng_Kiến_Phúc

[3] Lăng Tự Đức (lang_tam) — similarity 0.0313
    === Khu tẩm điện ===
Phía trước bên ngoài có 2 tòa nhà vuông Công Khiêm, Cung Khiêm. 3 dãy tam cấp bằng đá Thanh dẫn vào Khiêm Cung môn (謙宫門), là cửa chính của khu tẩm điện, 3 gian có cổ lâu. Bên tron...
    Nguồn: https://vi.wikipedia.org/wiki/Lăng_Tự_Đức



In [7]:
# Gõ không dấu: dense trượt, nhánh BM25 kéo địa danh đúng lên.
query = "lang tu duc co gi dep"
print("--- dense")
show(retriever.search(query, top_k=3), width=100)
print("--- hybrid")
show(hybrid.search(query, top_k=3), width=100)

--- dense
[1] Lăng Kiến Phúc (khac) — similarity 0.4422
    Lăng nằm ở bên tả ngạn hồ Lưu Khiêm, cùng phía với Khiêm Thọ Lăng. Bồi Lăng chia thành 2 phần: phần ...
    Nguồn: https://vi.wikipedia.org/wiki/Lăng_Kiến_Phúc

[2] Lăng Dục Đức (lang_tam) — similarity 0.4363
    === Khu tẩm điện ===...
    Nguồn: https://vi.wikipedia.org/wiki/Lăng_Dục_Đức

[3] Lăng Dục Đức (lang_tam) — similarity 0.4333
    == Chú thích ==...
    Nguồn: https://vi.wikipedia.org/wiki/Lăng_Dục_Đức

--- hybrid
[1] Lăng Tự Đức (lang_tam) — similarity 0.0308
    == Lăng lân cận ==...
    Nguồn: https://vi.wikipedia.org/wiki/Lăng_Tự_Đức

[2] Lăng Tự Đức (lang_tam) — similarity 0.0263
    === Lăng Khiêm Thọ ===...
    Nguồn: https://vi.wikipedia.org/wiki/Lăng_Tự_Đức

[3] Lăng Kiến Phúc (khac) — similarity 0.0262
    Lăng nằm ở bên tả ngạn hồ Lưu Khiêm, cùng phía với Khiêm Thọ Lăng. Bồi Lăng chia thành 2 phần: phần ...
    Nguồn: https://vi.wikipedia.org/wiki/Lăng_Kiến_Phúc



In [8]:
# Ngữ cảnh sẵn sàng đưa vào prompt của agent.
print(hybrid.build_rag_context(hybrid.search("Chùa Thiên Mụ được xây năm nào?", top_k=3)))

--- TRI THỨC ĐỊA DANH HUẾ (RAG CONTEXT) ---

[1] Địa danh: Chùa Thiên Mụ (chua)
Nội dung: Chùa Thiên Mụ hay còn gọi là chùa Linh Mụ là một ngôi chùa cổ nằm trên đồi Hà Khê, tả ngạn sông Hương, cách trung tâm thành phố Huế (Việt Nam) khoảng 5 km về phía tây. Chùa Thiên Mụ chính thức khởi lập năm Tân Sửu (1601), đời chúa Tiên Nguyễn Hoàng - vị chúa Nguyễn đầu tiên ở Đàng Trong.
Nguồn: https://vi.wikipedia.org/wiki/Chùa_Thiên_Mụ

[2] Địa danh: Chùa Thiên Mụ (chua)
Nội dung: Tháp Phước Duyên là một biểu tượng nổi tiếng gắn liền với chùa Thiên Mụ. Tháp cao 21 m, gồm 7 tầng, được xây dựng ở phía trước chùa vào năm 1844. Mỗi tầng tháp đều có thờ tượng Phật. Bên trong có cầu thang hình xoắn ốc dẫn lên tầng trên cùng, nơi trước đây có thờ tượng Phật bằng vàng. Phía trước tháp là đình Hương Nguyện, trên nóc đặt Pháp luân (bánh xe Phật pháp, biểu tượng Phật giáo. Pháp luân đặt trên đình Hương Nguyện quay khi gió thổi).
Trận bão năm 1904 đã tàn phá chùa nặng nề. Nhiều công trình bị hư hỏng, trong 

## 4. Benchmark truy hồi

Nhãn gán ở mức địa danh (QID Wikidata). Mọi chunk của một địa danh đúng được tính là đúng. Chỉ số chạy trên top-k chunk mà agent sẽ đưa vào prompt:

| Chỉ số | Ý nghĩa | Vì sao cần |
|---|---|---|
| Hit@k | Top-k có ít nhất một chunk của địa danh đúng | Chỉ số chính: LLM có thấy được nguồn đúng không |
| MRR@10 | Trung bình 1/hạng của chunk đúng đầu tiên | Chunk đúng nằm càng cao càng tốt |
| nDCG@5 | Chất lượng thứ hạng, tính mọi chunk đúng trong top-5 | Phân biệt "1 chunk đúng ở hạng 5" với "5 chunk đúng" |
| Precision@5 | Tỷ lệ chunk đúng trong 5 chunk đưa vào prompt | Chunk sai là nhiễu, dễ làm LLM trả lời sai |
| AnswerHit@5 | Top-5 có chunk thuộc địa danh đúng *và* chứa từ khoá câu trả lời | Đúng địa danh chưa đủ, phải đúng đoạn chứa đáp án |
| Recall@k (câu `multi`) | Tỷ lệ địa danh đúng xuất hiện trong top-k | Câu "các lăng vua Nguyễn" cần nhiều địa danh |
| Latency p50/p95 | Thời gian embed query + truy vấn SQL | Agent gọi tool này trong mỗi lượt hỏi đáp |

Retriever đứng đầu là **hybrid**, so với: **hybrid_fold** (bản hybrid chỉ bỏ dấu), **dense** và **bm25_multi** (từng nhánh riêng lẻ, tức ablation), **BM25** giữ dấu, **bm25_fold**, và **random** (mức sàn). Chạy thêm `python -m rag.benchmark --all-places` để đo tác động của việc loại category không phải điểm đến. Chênh lệch giữa hai retriever kèm khoảng tin cậy bootstrap 95%; khoảng không chứa 0 thì chênh lệch đáng tin với cỡ bộ câu hỏi hiện tại.

Câu hỏi mà địa danh đúng không có chunk nào bị loại khỏi số liệu và liệt kê riêng ở cuối báo cáo: đó là lỗ hổng dữ liệu, không phải lỗi truy hồi.

In [9]:
report, results = benchmark.run(conn, embedder=embedder, out_dir=None)
display(Markdown(report))

# Benchmark truy hồi RAG — 2026-09-27

- Bộ câu hỏi: `eval/rag_queries.yml`, 34 câu, 0 câu đã kiểm duyệt tay.
- Được chấm: 34 câu. Loại vì không có chunk nào của địa danh đúng: 0 câu.
- Kho: 1574 chunk, 1574 chunk có embedding của `BAAI/bge-m3`. Loại chunk của category: `boi_canh`, `ha_tang`, `don_vi_hanh_chinh`; còn 815 chunk để tìm.
- **Cảnh báo:** 34 câu chưa kiểm duyệt tay. Chưa dùng số liệu này trong báo cáo đồ án.

## Tổng hợp

| Retriever | Hit@1 | Hit@3 | Hit@5 | Hit@10 | MRR@10 | nDCG@5 | P@5 | AnswerHit@5 | p50 ms | p95 ms |
|---|---|---|---|---|---|---|---|---|---|---|
| hybrid:BAAI/bge-m3+bm25_multi | 0.853 | 0.971 | 0.971 | 1.000 | 0.912 | 0.733 | 0.700 | 1.000 | 566.4 | 905.7 |
| hybrid:BAAI/bge-m3+bm25_fold | 0.824 | 0.941 | 1.000 | 1.000 | 0.892 | 0.708 | 0.682 | 1.000 | 537.0 | 1261.9 |
| dense:BAAI/bge-m3 | 0.735 | 0.853 | 0.853 | 0.971 | 0.805 | 0.613 | 0.582 | 1.000 | 675.6 | 1113.7 |
| bm25 | 0.765 | 0.912 | 0.912 | 0.971 | 0.830 | 0.614 | 0.571 | 1.000 | 3.2 | 6.3 |
| bm25_fold | 0.794 | 0.882 | 0.912 | 1.000 | 0.858 | 0.623 | 0.576 | 0.750 | 4.1 | 6.7 |
| bm25_multi | 0.882 | 0.912 | 0.912 | 1.000 | 0.906 | 0.693 | 0.653 | 0.875 | 8.9 | 13.7 |
| random | 0.000 | 0.088 | 0.176 | 0.206 | 0.058 | 0.036 | 0.047 | 0.000 | 0.1 | 0.4 |

AnswerHit tính trên 8 câu có `answer_keywords`.

## Theo loại câu hỏi (Hit@5 / MRR)

| Retriever | descriptive | fact | multi | no_diacritics | paraphrase |
|---|---|---|---|---|---|
| hybrid:BAAI/bge-m3+bm25_multi | 1.000 / 1.000 (n=4) | 1.000 / 1.000 (n=9) | 0.667 / 0.556 (n=3) | 1.000 / 0.875 (n=4) | 1.000 / 0.917 (n=14) |
| hybrid:BAAI/bge-m3+bm25_fold | 1.000 / 1.000 (n=4) | 1.000 / 1.000 (n=9) | 1.000 / 0.500 (n=3) | 1.000 / 1.000 (n=4) | 1.000 / 0.845 (n=14) |
| dense:BAAI/bge-m3 | 1.000 / 1.000 (n=4) | 1.000 / 1.000 (n=9) | 0.667 / 0.375 (n=3) | 0.000 / 0.105 (n=4) | 1.000 / 0.917 (n=14) |
| bm25 | 1.000 / 1.000 (n=4) | 1.000 / 0.926 (n=9) | 1.000 / 1.000 (n=3) | 0.500 / 0.358 (n=4) | 0.929 / 0.818 (n=14) |
| bm25_fold | 1.000 / 0.875 (n=4) | 0.778 / 0.808 (n=9) | 1.000 / 1.000 (n=3) | 1.000 / 1.000 (n=4) | 0.929 / 0.815 (n=14) |
| bm25_multi | 1.000 / 1.000 (n=4) | 0.889 / 0.831 (n=9) | 1.000 / 1.000 (n=3) | 1.000 / 1.000 (n=4) | 0.857 / 0.881 (n=14) |
| random | 0.000 / 0.000 (n=4) | 0.111 / 0.037 (n=9) | 0.333 / 0.159 (n=3) | 0.250 / 0.062 (n=4) | 0.214 / 0.064 (n=14) |

## Recall@k cho câu nhiều địa danh (`multi`)

| Retriever | Recall@1 | Recall@3 | Recall@5 | Recall@10 |
|---|---|---|---|---|
| hybrid:BAAI/bge-m3+bm25_multi | 0.067 | 0.242 | 0.392 | 0.653 |
| hybrid:BAAI/bge-m3+bm25_fold | 0.067 | 0.200 | 0.461 | 0.586 |
| dense:BAAI/bge-m3 | 0.000 | 0.150 | 0.258 | 0.478 |
| bm25 | 0.219 | 0.439 | 0.589 | 0.808 |
| bm25_fold | 0.219 | 0.439 | 0.506 | 0.547 |
| bm25_multi | 0.219 | 0.369 | 0.436 | 0.767 |
| random | 0.000 | 0.042 | 0.042 | 0.258 |

## So sánh cặp (bootstrap 95%, lấy mẫu lại theo câu hỏi)

| So sánh | Chỉ số | Chênh lệch | CI 95% |
|---|---|---|---|
| hybrid:BAAI/bge-m3+bm25_multi − hybrid:BAAI/bge-m3+bm25_fold | mrr | +0.020 | [-0.037, +0.081] |
| hybrid:BAAI/bge-m3+bm25_multi − hybrid:BAAI/bge-m3+bm25_fold | hit@5 | -0.029 | [-0.088, +0.000] |
| hybrid:BAAI/bge-m3+bm25_multi − dense:BAAI/bge-m3 | mrr | +0.107 | [+0.025, +0.203] |
| hybrid:BAAI/bge-m3+bm25_multi − dense:BAAI/bge-m3 | hit@5 | +0.118 | [+0.029, +0.235] |
| hybrid:BAAI/bge-m3+bm25_multi − bm25 | mrr | +0.082 | [-0.029, +0.200] |
| hybrid:BAAI/bge-m3+bm25_multi − bm25 | hit@5 | +0.059 | [-0.059, +0.176] |
| hybrid:BAAI/bge-m3+bm25_multi − bm25_fold | mrr | +0.053 | [-0.067, +0.172] |
| hybrid:BAAI/bge-m3+bm25_multi − bm25_fold | hit@5 | +0.059 | [-0.059, +0.176] |
| hybrid:BAAI/bge-m3+bm25_multi − bm25_multi | mrr | +0.006 | [-0.098, +0.109] |
| hybrid:BAAI/bge-m3+bm25_multi − bm25_multi | hit@5 | +0.059 | [-0.059, +0.176] |
| hybrid:BAAI/bge-m3+bm25_multi − random | mrr | +0.854 | [+0.758, +0.935] |
| hybrid:BAAI/bge-m3+bm25_multi − random | hit@5 | +0.794 | [+0.647, +0.912] |

## Câu trượt top-5 (phân tích lỗi)

### hybrid:BAAI/bge-m3+bm25_multi: 1 câu
- `q030` (multi) "Các miếu thờ trong Hoàng thành Huế" → top-3: Hoàng thành Huế

### hybrid:BAAI/bge-m3+bm25_fold: 0 câu

### dense:BAAI/bge-m3: 5 câu
- `q030` (multi) "Các miếu thờ trong Hoàng thành Huế" → top-3: Hoàng thành Huế
- `q031` (no_diacritics) "lang tu duc co gi dep" → top-3: Lăng Kiến Phúc, Lăng Dục Đức
- `q032` (no_diacritics) "chua thien mu xay nam nao" → top-3: Đàn Nam Giao (triều Nguyễn), Rú Chá
- `q033` (no_diacritics) "cau truong tien dai bao nhieu" → top-3: Hoàng thành Huế, Bảo tàng Mỹ thuật Cung đình Huế, Chùa Thiên Mụ
- `q034` (no_diacritics) "dai noi hue gom nhung cong trinh nao" → top-3: Lăng Đồng Khánh, Đàn Nam Giao (triều Nguyễn), Cồn Dã Viên

### bm25: 3 câu
- `q026` (paraphrase) "Tòa thành bao quanh Hoàng thành Huế xây theo kiểu Vauban" → top-3: Cố đô Huế, Điện Thái Hòa (hoàng thành Huế)
- `q031` (no_diacritics) "lang tu duc co gi dep" → top-3: Hoàng thành Huế, Cầu Trường Tiền
- `q033` (no_diacritics) "cau truong tien dai bao nhieu" → top-3: Hoàng thành Huế, Lăng Gia Long, Cố đô Huế

### bm25_fold: 3 câu
- `q004` (fact) "Lăng vua Tự Đức còn có tên gọi khác là gì?" → top-3: Cồn Dã Viên, Kinh thành Huế, Sông Bồ
- `q009` (fact) "Lăng Gia Long có tên gọi là gì?" → top-3: Kinh thành Huế, Cầu Dã Viên, Lăng Khải Định
- `q026` (paraphrase) "Tòa thành bao quanh Hoàng thành Huế xây theo kiểu Vauban" → top-3: Hoàng thành Huế, Cố đô Huế, Điện Thái Hòa (hoàng thành Huế)

### bm25_multi: 3 câu
- `q004` (fact) "Lăng vua Tự Đức còn có tên gọi khác là gì?" → top-3: Sông Hương, Lăng Thiệu Trị, Sông Bồ
- `q018` (paraphrase) "Cây cầu sắt bắc qua sông Hương xây thời Pháp thuộc, có sáu nhịp" → top-3: Cầu Dã Viên, Sông Hương
- `q026` (paraphrase) "Tòa thành bao quanh Hoàng thành Huế xây theo kiểu Vauban" → top-3: Điện Thái Hòa (hoàng thành Huế), Cố đô Huế, Chùa Từ Hiếu

### random: 28 câu
- `q002` (fact) "Tháp Phước Duyên ở chùa Thiên Mụ có mấy tầng?" → top-3: Lăng Minh Mạng, Hoàng thành Huế, Văn miếu Huế
- `q003` (paraphrase) "Ngôi chùa cổ có tháp bảy tầng nằm trên đồi bên bờ sông Hương" → top-3: Thái Tổ Miếu (hoàng thành Huế), Lăng Tự Đức, Cầu Tam Giang
- `q004` (fact) "Lăng vua Tự Đức còn có tên gọi khác là gì?" → top-3: Thế Tổ Miếu, Cửu Đỉnh của nhà Nguyễn, Cầu ngói Thanh Toàn
- `q005` (paraphrase) "Lăng của vị vua làm nhiều thơ, có hồ Lưu Khiêm và Xung Khiêm tạ" → top-3: Điện Long An, Cầu Trường Tiền, Nghênh Lương đình
- `q006` (fact) "Lăng Khải Định còn gọi là gì?" → top-3: Ngọ Môn (hoàng thành Huế), Cố đô Huế
- `q008` (fact) "Tên chính thức của lăng vua Minh Mạng" → top-3: Chùa Diệu Đế, Lăng Thiệu Trị, Lăng Đồng Khánh
- `q009` (fact) "Lăng Gia Long có tên gọi là gì?" → top-3: Chợ Đông Ba, Chùa Từ Hiếu, Ngọ Môn (hoàng thành Huế)
- `q010` (fact) "Lăng Thiệu Trị còn được gọi là gì?" → top-3: Cửu Đỉnh của nhà Nguyễn, Lầu Tứ Phương Vô Sự, Cố đô Huế
- `q011` (fact) "Lăng Đồng Khánh có tên chữ là gì?" → top-3: Hoàng thành Huế, Chùa Quốc Ân, Lăng Tự Đức
- `q015` (paraphrase) "Chín cái đỉnh đồng đúc thời Minh Mạng đặt trước Thế Miếu" → top-3: Ngọ Môn (hoàng thành Huế), Cầu ngói Thanh Toàn, Cố đô Huế
- `q016` (fact) "Bảo tàng Cổ vật Cung đình Huế đặt trong tòa điện nào?" → top-3: Cầu Phú Xuân, Chùa Quốc Ân, Cầu Tam Giang
- `q017` (paraphrase) "Nơi vua nhà Nguyễn làm lễ tế trời đất" → top-3: Lăng Tự Đức, Hoàng thành Huế, Cầu ngói Thanh Toàn
- `q018` (paraphrase) "Cây cầu sắt bắc qua sông Hương xây thời Pháp thuộc, có sáu nhịp" → top-3: Lăng Đồng Khánh, Chùa Từ Đàm, Bảo tàng Mỹ thuật Cung đình Huế
- `q019` (descriptive) "Giới thiệu về chợ Đông Ba" → top-3: Sông Hương, Lăng Dục Đức, Hoàng thành Huế
- `q020` (paraphrase) "Ngôi chùa ở Huế gắn với các thái giám triều Nguyễn" → top-3: Kinh thành Huế, Bảo tàng Mỹ thuật Cung đình Huế, Chợ Đông Ba
- `q021` (descriptive) "Chùa Từ Đàm có vai trò gì trong lịch sử Phật giáo Việt Nam?" → top-3: Cầu Trường Tiền, Lăng Gia Long, Nhà thờ Dòng Chúa Cứu Thế, Huế
- `q022` (paraphrase) "Cung điện của vua Khải Định khi còn là thái tử, kiến trúc pha phương Tây" → top-3: Đàn Nam Giao (triều Nguyễn), Hoàng thành Huế, Lăng Dục Đức
- `q023` (descriptive) "Nhà thờ chính tòa Phủ Cam được xây dựng như thế nào?" → top-3: Cồn Dã Viên, Cầu ngói Thanh Toàn, Cửu Đỉnh của nhà Nguyễn
- `q024` (paraphrase) "Nơi thờ Khổng Tử và dựng bia tiến sĩ triều Nguyễn ở Huế" → top-3: Lăng Gia Long, Điện Thái Hòa (hoàng thành Huế), Khách sạn Saigon Morin
- `q025` (paraphrase) "Cửa biển nơi phá Tam Giang thông ra biển Đông" → top-3: Hoàng thành Huế, Chùa Từ Hiếu
- `q026` (paraphrase) "Tòa thành bao quanh Hoàng thành Huế xây theo kiểu Vauban" → top-3: Văn miếu Huế, Công viên nước Hồ Thủy Tiên, Cửu Đỉnh của nhà Nguyễn
- `q027` (paraphrase) "Cây cầu có mái ngói cổ ở làng quê gần Huế" → top-3: Nghênh Lương đình, Khách sạn Saigon Morin, Hoàng thành Huế
- `q028` (descriptive) "Sông Hương bắt nguồn từ đâu?" → top-3: Sông Bồ, Lăng Đồng Khánh, Lăng Khải Định
- `q029` (multi) "Các ngôi chùa nổi tiếng ở Huế" → top-3: Lăng Tự Đức, Kinh thành Huế, Cầu Trường Tiền
- `q030` (multi) "Các miếu thờ trong Hoàng thành Huế" → top-3: Chùa Thiên Mụ, Lăng Tự Đức, Điện Long An
- `q031` (no_diacritics) "lang tu duc co gi dep" → top-3: The Dreamers, Hoàng thành Huế, Cồn Dã Viên
- `q032` (no_diacritics) "chua thien mu xay nam nao" → top-3: Lăng Thiệu Trị, Lăng Tự Đức, Khách sạn Saigon Morin
- `q033` (no_diacritics) "cau truong tien dai bao nhieu" → top-3: Lầu Tứ Phương Vô Sự, Khách sạn Saigon Morin, Kinh thành Huế

## Lỗ hổng dữ liệu

- `q025`: QID không có trong DB: Q10825419


## 5. Phân tích lỗi: hybrid so với dense theo từng câu

In [10]:
by_query = {
    name: {r.query.id: r for r in rs} for name, rs in results.items()
}
a_name, b_name = hybrid.name, retriever.name
print(f"{'id':6} {'type':14} {'hybrid':>6} {'dense':>6}  query")
for qid, a in by_query[a_name].items():
    b = by_query[b_name][qid]
    if a.metrics["mrr"] != b.metrics["mrr"]:
        print(f"{qid:6} {a.query.type:14} {a.metrics['mrr']:6.2f} {b.metrics['mrr']:6.2f}  {a.query.query}")

id     type           hybrid  dense  query
q018   paraphrase       0.33   0.50  Cây cầu sắt bắc qua sông Hương xây thời Pháp thuộc, có sáu nhịp
q022   paraphrase       0.50   0.33  Cung điện của vua Khải Định khi còn là thái tử, kiến trúc pha phương Tây
q029   multi            1.00   0.50  Các ngôi chùa nổi tiếng ở Huế
q030   multi            0.17   0.12  Các miếu thờ trong Hoàng thành Huế
q031   no_diacritics    1.00   0.17  lang tu duc co gi dep
q032   no_diacritics    1.00   0.14  chua thien mu xay nam nao
q033   no_diacritics    1.00   0.00  cau truong tien dai bao nhieu
q034   no_diacritics    0.50   0.11  dai noi hue gom nhung cong trinh nao


In [11]:
# Xem top-5 của một câu cụ thể.
qid = "q005"
for name in results:
    r = by_query[name][qid]
    print(f"--- {name}: hit@5={r.metrics['hit@5']:.0f}, mrr={r.metrics['mrr']:.2f}")
    for c in r.chunks[:5]:
        print(f"    {c.place_name}: {c.content[:90]}...")

--- hybrid:BAAI/bge-m3+bm25_multi: hit@5=1, mrr=1.00
    Lăng Tự Đức: Bố cục khu lăng gồm 2 phần chính, bố trí trên 2 trục lăng mộ và tẩm điện dọc song song với...
    Lăng Tự Đức: === Khu tẩm điện ===
Phía trước bên ngoài có 2 tòa nhà vuông Công Khiêm, Cung Khiêm. 3 dãy...
    Lăng Kiến Phúc: Lăng Kiến Phúc, hay Bồi Lăng (陪陵), là lăng tẩm của hoàng đế Kiến Phúc, vị hoàng đế thứ 7 c...
    Lăng Kiến Phúc: Lăng nằm ở bên tả ngạn hồ Lưu Khiêm, cùng phía với Khiêm Thọ Lăng. Bồi Lăng chia thành 2 p...
    Lăng Thiệu Trị: === Khu tẩm thờ ===
Khu vực điện thờ được xây dựng cách lầu Đức Hinh 100m về phía trái. Mở...
--- hybrid:BAAI/bge-m3+bm25_fold: hit@5=1, mrr=1.00
    Lăng Tự Đức: Bố cục khu lăng gồm 2 phần chính, bố trí trên 2 trục lăng mộ và tẩm điện dọc song song với...
    Lăng Tự Đức: === Khu tẩm điện ===
Phía trước bên ngoài có 2 tòa nhà vuông Công Khiêm, Cung Khiêm. 3 dãy...
    Lăng Kiến Phúc: Lăng Kiến Phúc, hay Bồi Lăng (陪陵), là lăng tẩm của hoàng đế Kiến Phúc, vị hoàng đế thứ 7 

## 6. Lưu báo cáo

Ghi `data/qa/rag_benchmark_<ngày>.md` và `.csv` (từng câu, từng retriever) để đưa vào báo cáo đồ án. Terminal tương đương: `python -m rag.benchmark`.

In [12]:
report, _ = benchmark.run(conn, embedder=embedder)
print("Đã ghi data/qa/rag_benchmark_*.md và .csv")

Đã ghi data/qa/rag_benchmark_*.md và .csv


In [13]:
conn.close()